In [ ]:
%matplotlib inline
import flopy
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pathlib as pl
import pandas as pd
import sys

In [ ]:
sys.path.append("../common")
from liss_settings import (
    cx, cx_provider, extent, boxx, boxy, extentmax, fig_ext, transparent,
    get_scenario_name, get_results_path, get_modflow_grid_name,
)


In [ ]:
units = "mm"
conversion_factor = 1.0
if units == "mm":
    conversion_factor = 25.4
total_key = f"total_{units}"

## Load base MODFLOW model

In [ ]:
# ---- scenario family ---------------------------------------------------------
# This notebook compares the SAME model at different MODFLOW <-> D-Flow FM
# coupling frequencies, so resolution and connection count are held fixed and the
# coupling frequency is what varies.
domain = "gp"
boundary_condition = "chd"
resolution = "high"
n_connections = 244                    # connections ACTUALLY resolved by step2
# ------------------------------------------------------------------------------

mf_grid_name = get_modflow_grid_name(domain=domain, boundary_condition=boundary_condition)
ws = f"../modflow/{mf_grid_name}/base/"
sim = flopy.mf6.MFSimulation.load(sim_ws=ws, verbosity_level=0)

gwf = sim.get_model()

# The coupled coastal boundary is split across two packages in the gp_chd model:
# GHB carries the "inner" bays and the chd_surface CHD package carries "coastal"
# and "perimeter". The old greenport model put all three (SOUND / INNER /
# PECONIC) in GHB, hence the single nghb area used previously.
nghb = gwf.ghb.stress_period_data.get_dataframe()[0].shape[0]
nchd = gwf.get_package("chd_surface").stress_period_data.get_dataframe()[0].shape[0]

# DIS is in feet with 500 ft cells, so area is ft^2 and flux/area is ft/d;
# the *12.0 below converts to in/d and conversion_factor to mm/d.
cell_area = 500.0 * 500.0
area = (nghb + nchd) * cell_area

print(f"model {mf_grid_name}: {nghb} GHB + {nchd} CHD(surface) = {nghb + nchd} "
      f"coupled boundary cells")
print(f"normalizing area: {area:,.0f} ft^2")


## Load the coastal boundary observation data

In [ ]:
# Coupling frequencies to compare, in hours. Each becomes a scenario directory
# written by step2_run_coupled_models; any that have not been run are reported
# and skipped rather than raising.
couple_freqs = [24.0, 8.0, 4.0, 2.0, 1.0, 0.5, 0.25]

sim_dirs = [
    get_scenario_name(domain, resolution, f, n_connections) for f in couple_freqs
]
sim_dirs


In [ ]:
sim_dict = {}
for freq, scenario in zip(couple_freqs, sim_dirs):
    sim_dict[scenario] = {
        "freq": freq,
        "ws": get_results_path(domain, resolution, freq, n_connections),
    }

# Drop scenarios that have not been run yet so the comparison still plots.
missing = [k for k, v in sim_dict.items() if not (v["ws"] / "gwf.ghb.obs.csv").is_file()]
for k in missing:
    print(f"  skipping {k} - not run yet ({sim_dict[k]['ws']})")
    del sim_dict[k]
sim_dirs = [k for k in sim_dirs if k in sim_dict]
couple_freqs = [sim_dict[k]["freq"] for k in sim_dirs]
assert sim_dirs, "no scenarios have been run - run step2_run_coupled_models first"
sim_dirs


In [ ]:
# Boundary flow terms that make up the coastal exchange. INNER comes from the GHB
# obs; COASTAL and PERIMETER come from the chd_surface obs. All are driven by
# D-Flow FM water levels, but PERIMETER is the lateral regional boundary rather
# than the coast, and it dominates the sum (mean +3.2e6 vs +1.7e6 ft3/d), so it
# is excluded by default -- add it back below to count it.
#
# Sign convention: MF6 reports boundary flow positive INTO the model, and the
# terms are negated here so positive = discharge OUT of the aquifer. In this
# density-coupled (BUY) model COASTAL is net INTO the aquifer, so the cumulative
# curve runs negative -- unlike the older greenport GHB-only model, where it was
# net discharge. That is a model difference, not a units error.
ghb_terms = ["INNER"]
chd_terms = ["COASTAL"]          # add "PERIMETER" to include the lateral boundary

for key, value in sim_dict.items():
    ghb = flopy.utils.Mf6Obs(value["ws"] / "gwf.ghb.obs.csv").get_dataframe(
        start_datetime="1-1-2010")
    chd = flopy.utils.Mf6Obs(value["ws"] / "gwf.chd.obs.csv").get_dataframe(
        start_datetime="1-1-2010")

    df = ghb.join(chd.drop(columns=["totim"], errors="ignore"), rsuffix="_chd")
    df["delt"] = df["totim"].diff()
    df.loc[df["delt"].isnull(), "delt"] = df["delt"].iloc[1]
    df.drop("totim", axis=1, inplace=True)

    df["TOTAL"] = 0.0
    for col_name in ghb_terms + chd_terms:
        assert col_name in df.columns, f"{key}: {col_name} missing from {list(df.columns)}"
        df["TOTAL"] += -conversion_factor * 12.0 * df[col_name] / area
    df["CUM_TOTAL"] = df["TOTAL"].cumsum() * df["delt"]
    df["ZERO"] = 0.0
    df.drop("delt", axis=1, inplace=True)

    sim_dict[key]["ntimes"] = df.shape[0]
    sim_dict[key]["df"] = df.copy()
    sim_dict[key][total_key] = df["CUM_TOTAL"].iloc[-1]
    print(f"  {key}: {df.shape[0]} times, cumulative {df['CUM_TOTAL'].iloc[-1]:.2f} {units}")


## Plot the coastal exchange

In [ ]:
ws = pl.Path("figures")
ws.mkdir(exist_ok=True, parents=True)

In [ ]:
# label each scenario by its coupling interval, e.g. " 8 Hour"
labels = []
for freq in couple_freqs:
    if freq == 24.0:
        labels.append(f"{freq / 24:>2.0f} Day ")
    elif freq >= 1.0:
        labels.append(f"{freq:>2.0f} Hour")
    else:
        labels.append(f"{freq * 60:>2.0f} Min.")
labels


In [ ]:
line_styles = ["-", "--", "-.", ":", (0, (3, 10, 1, 10, 1, 10)), (0, (3, 1, 1, 1, 1, 1))]

In [ ]:
colors = [value for key, value in mcolors.TABLEAU_COLORS.items()]

In [ ]:
cum_labels = []
for idx, value in enumerate(sim_dirs):
    total_flux = sim_dict[value][total_key]
    cum_labels.append(f"{labels[idx]} ({total_flux:5.2f} {units})")
cum_labels


In [ ]:
# Axis limits derived from the data. The previous hard-coded ylim(0, 15*cf)
# assumed a positive cumulative curve and silently clipped everything when the
# sign came out negative.
_rate = np.concatenate([sim_dict[k]["df"]["TOTAL"].to_numpy() for k in sim_dirs])
_cum = np.concatenate([sim_dict[k]["df"]["CUM_TOTAL"].to_numpy() for k in sim_dirs])

def _pad(lo, hi, frac=0.08):
    if lo == hi:
        lo, hi = lo - 1.0, hi + 1.0
    m = frac * (hi - lo)
    return lo - m, hi + m

rate_ylim = _pad(float(np.nanmin(_rate)), float(np.nanmax(_rate)))
cum_ylim = _pad(min(0.0, float(np.nanmin(_cum))), max(0.0, float(np.nanmax(_cum))))
print(f"rate ylim {rate_ylim[0]:.2f} .. {rate_ylim[1]:.2f} {units}/d")
print(f"cum  ylim {cum_ylim[0]:.2f} .. {cum_ylim[1]:.2f} {units}")

In [ ]:
with flopy.plot.styles.USGSMap():
    fig, axs = plt.subplots(
        ncols=1,
        nrows=len(labels) + 1,
        layout="constrained",
        figsize=(7.5, 2 * (len(labels) + 1)),
        )
    for idx in range(len(labels)):
        ax = axs[idx]
        ax.set_ylim(*rate_ylim)
        df = sim_dict[sim_dirs[idx]]["df"]
        df["TOTAL"].plot(ax=ax, lw=0.75, ls="-", color=colors[idx], sharex=True)
        df["ZERO"].plot(ax=ax, lw=0.5, ls="--", color="black", sharex=True)
        ax.set_ylabel(f"Coastal\nexchange, {units}")
        ax.set_xlabel("")
        flopy.plot.styles.heading(ax, idx=idx, heading=labels[idx])

    ax = axs[-1]
    ax.set_ylim(*cum_ylim)
    for idx, value in enumerate(sim_dirs):
        sim_dict[value]["df"]["CUM_TOTAL"].plot(ax=ax, lw=0.75, ls=line_styles[idx % len(line_styles)], color=colors[idx], label=cum_labels[idx])
    ax.set_ylabel(f"Cumulative coastal\nexchange, {units}")
    ax.set_xlabel("")
    flopy.plot.styles.heading(ax, idx=len(labels), heading="Cumulative")
    
    leg = flopy.plot.styles.graph_legend(ax=ax, loc="upper left", title="none", ncol=2)

    fig.savefig(ws / f"coastal_flux_summary_{resolution}{fig_ext}", dpi=300, transparent=transparent)


In [ ]:
# first and (if available) second-to-last scenario
_i2 = -2 if len(sim_dirs) > 1 else 0
plot_sim_dirs = [sim_dirs[0], sim_dirs[_i2]]
plot_labels = [labels[0], labels[_i2]]

with flopy.plot.styles.USGSMap():
    fig, axs = plt.subplots(
        ncols=1,
        nrows=len(plot_labels) + 1,
        layout="constrained",
        figsize=(11, 2 * (len(plot_labels) + 1)),
        )
    for idx in range(len(plot_labels)):
        ax = axs[idx]
        ax.set_ylim(*rate_ylim)
        df = sim_dict[plot_sim_dirs[idx]]["df"]
        df["TOTAL"].plot(ax=ax, lw=0.75, ls="-", color=colors[idx % len(colors)], sharex=True)
        df["ZERO"].plot(ax=ax, lw=0.5, ls="--", color="black", sharex=True)
        ax.set_ylabel(f"Coastal\nexchange, {units}")
        ax.set_xlabel("")
        flopy.plot.styles.heading(ax, idx=idx, heading=plot_labels[idx])

    ax = axs[-1]
    ax.set_ylim(*cum_ylim)
    for idx, value in enumerate(sim_dirs):
        sim_dict[value]["df"]["CUM_TOTAL"].plot(ax=ax, lw=0.75, color=colors[idx % len(colors)], label=cum_labels[idx])
    ax.set_ylabel(f"Cumulative coastal\nexchange, {units}")
    ax.set_xlabel("")
    flopy.plot.styles.heading(ax, idx=len(plot_labels), heading="Cumulative")
    
    leg = flopy.plot.styles.graph_legend(ax=ax, loc="upper left", title="none", ncol=2)

    fig.savefig(ws / f"coastal_flux_poster_summary_{resolution}{fig_ext}", dpi=300, transparent=transparent)

In [ ]:
with flopy.plot.styles.USGSMap():
    fig, ax = plt.subplots(
        ncols=1,
        nrows=1,
        layout="constrained",
        figsize=(11, 3),
        )

    ax.set_ylim(*cum_ylim)
    for idx, value in enumerate(sim_dirs):
        sim_dict[value]["df"]["CUM_TOTAL"].plot(ax=ax, lw=0.75, color=colors[idx % len(colors)], label=cum_labels[idx])
    ax.set_ylabel(f"Cumulative coastal\nexchange, {units}")
    ax.set_xlabel("")
    
    leg = flopy.plot.styles.graph_legend(ax=ax, loc="upper left", title="none", ncol=2)

    fig.savefig(ws / f"coastal_cumulative_flux_{resolution}{fig_ext}", dpi=300, transparent=transparent)